In [ ]:
import os, subprocess

def sh(cmd, check=True):
    """Run a shell command and stream its output into the notebook."""
    print(f"\n$ {cmd}")
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")
    return p.returncode

REPO = "/content/executorch_on_arm"
VENV = f"{REPO}/.cpu_aot_venv"
PY = f"{VENV}/bin/python"          # always use this, never plain `python`
WANT = "3.12.14"

# 1) System packages
sh("sudo apt-get update -qq")
sh("sudo apt-get install -y -qq git curl ca-certificates build-essential "
   "libssl-dev zlib1g-dev libbz2-dev libreadline-dev libsqlite3-dev "
   "libncurses-dev xz-utils tk-dev libxml2-dev libxmlsec1-dev libffi-dev liblzma-dev")

# 2) Clone the course repo
if not os.path.isdir(REPO):
    sh(f"git clone https://github.com/ArmDeveloperEcosystem/arm-courses-code-executorch.git {REPO}")
os.chdir(REPO)

# 3) Create the venv with exactly Python 3.12.14 (uv first, pyenv as fallback)
sh("pip install -q uv")
sh(f"rm -rf {VENV}")
ok = sh(f"uv python install {WANT} && uv venv --python {WANT} {VENV}", check=False) == 0

if not ok:
    print("\nuv could not provide 3.12.14, building it with pyenv (takes several minutes)...")
    sh("curl -fsSL https://pyenv.run | bash")
    os.environ["PYENV_ROOT"] = os.path.expanduser("~/.pyenv")
    os.environ["PATH"] = f"{os.environ['PYENV_ROOT']}/bin:{os.environ['PATH']}"
    sh(f"rm -rf {VENV}")
    sh(f"pyenv install -s {WANT} || (pyenv update && pyenv install -s {WANT})")
    sh(f"{os.environ['PYENV_ROOT']}/versions/{WANT}/bin/python -m venv {VENV}")

# 4) Verify the version BEFORE installing anything
ver = subprocess.check_output([PY, "--version"], text=True).strip()
print("\nVenv Python:", ver)
assert ver == f"Python {WANT}", f"Wrong Python version: {ver}"

# 5) Install the pinned Module 1 requirements
sh(f"uv pip install --python {PY} -r module-1-transformer-inference-with-xnnpack/requirements.txt")


# 6) Optional: Hugging Face login (only needed for gated Llama 3.2)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    sh(f"{PY} -c \"import os; from huggingface_hub import login; login(os.environ['HF_TOKEN'])\"")
except Exception as e:
    print("\nSkipping HF login (add HF_TOKEN under the Secrets key icon for Llama):", e)

# 7) Optional: mount Drive and define a helper to save outputs
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted:", e)

def save_outputs(pattern="*.pte tokenizer*", dest="/content/drive/MyDrive/executorch_outputs"):
    os.makedirs(dest, exist_ok=True)
    sh(f"cp -v {pattern} {dest}/", check=False)

print("\nSetup complete. Run scripts with:  sh(f'{PY} module-1-transformer-inference-with-xnnpack/<script>.py')")


$ sudo apt-get update -qq
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)

$ sudo apt-get install -y -qq git curl ca-certificates build-essential libssl-dev zlib1g-dev libbz2-dev libreadline-dev libsqlite3-dev libncurses-dev xz-utils tk-dev libxml2-dev libxmlsec1-dev libffi-dev liblzma-dev
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 25.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
(Reading database ... 
(Reading database ... 5%
(Reading database ... 10%
(Reading database ... 15%
(Reading dat

In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
sh(f"{PY} -c \"import os; from huggingface_hub import login; login(os.environ['HF_TOKEN'])\"")


$ /content/executorch_on_arm/.cpu_aot_venv/bin/python -c "import os; from huggingface_hub import login; login(os.environ['HF_TOKEN'])"
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


0

In [ ]:
sh(f"{PY} -c \"from huggingface_hub import whoami; print(whoami()['name'])\"")


$ /content/executorch_on_arm/.cpu_aot_venv/bin/python -c "from huggingface_hub import whoami; print(whoami()['name'])"
lingesh007


0

In [ ]:
import subprocess

def sh(cmd, check=True, quiet=False):
    if not quiet:
        print(f"\n$ {cmd}")
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not quiet:
            print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}")
    return p.returncode

In [ ]:
import os, subprocess, re

# ===== config =====
ET_VER   = "1.3.1"                       # <- version from your team's VERSIONS.md
CTX      = 1024                          # context length (512 or 1024 for the phone)
GROUP    = 32                            # quantization group size
LREPO    = "/content/executorch_src"
LVENV    = "/content/.llama_venv"
LPY      = f"{LVENV}/bin/python"
WDIR     = "/content/llama32_1b"
OUT      = "/content/export"
PTE_NAME = f"llama3_2_1b_8da4w_{CTX}.pte"
ENVP     = f"export PATH={LVENV}/bin:$PATH VIRTUAL_ENV={LVENV}; cd {LREPO};"

assert os.environ.get("HF_TOKEN"), "HF_TOKEN missing: run the setup cell first (needs Colab Secret + Notebook access)."

# ===== 1) find the git tag matching your release =====
tags = subprocess.check_output(
    "git ls-remote --tags https://github.com/pytorch/executorch.git | awk '{print $2}'",
    shell=True, text=True).splitlines()
cands = [t.replace("refs/tags/", "") for t in tags if re.fullmatch(rf"refs/tags/v?{re.escape(ET_VER)}", t)]
assert cands, f"No tag found for {ET_VER}. Check the release name and set ET_VER."
TAG = cands[0]
print("Using tag:", TAG)

# ===== 2) clone ExecuTorch at that tag =====
if not os.path.isdir(LREPO):
    sh(f"git clone --branch {TAG} --depth 1 --recurse-submodules --shallow-submodules "
       f"https://github.com/pytorch/executorch.git {LREPO}")
else:
    print("ExecuTorch source: already cloned.")

# ===== 3) separate venv + install from source (slow the first time) =====
if not os.path.exists(LPY):
    sh("pip install -q uv", quiet=True)
    sh(f"uv venv --seed --python 3.12.14 {LVENV}")
if subprocess.run(f"{LPY} -c 'import executorch'", shell=True, capture_output=True).returncode != 0:
    sh(f"{ENVP} ./install_executorch.sh")
    sh(f"{ENVP} bash examples/models/llama/install_requirements.sh")
else:
    print("Llama venv: ExecuTorch already installed.")
sh(f"{LPY} -c 'import executorch; print(\"executorch\", executorch.__version__)'")

# ===== 4) download original-format Llama 3.2 1B Instruct weights =====
CK = f"{WDIR}/original"
if not os.path.exists(f"{CK}/consolidated.00.pth"):
    sh(f"{PY} -c \"from huggingface_hub import snapshot_download; "
       f"snapshot_download('meta-llama/Llama-3.2-1B-Instruct', allow_patterns=['original/*'], local_dir='{WDIR}')\"")
else:
    print("Weights: already downloaded.")
sh(f"ls -lh {CK}")

# ===== 5) verify the flags exist in THIS release before exporting =====
help_txt = subprocess.run(f"{ENVP} python -m examples.models.llama.export_llama --help",
                          shell=True, capture_output=True, text=True).stdout
need = ["--model", "--checkpoint", "--params", "-kv", "--use_sdpa_with_kv_cache", "--xnnpack",
        "--quantization_mode", "--group_size", "--embedding-quantize", "--max_seq_length",
        "--metadata", "--output_name"]
missing = [f for f in need if f not in help_txt]
if missing:
    print("These flags are NOT in --help for this release:", missing)
    print("Open examples/models/llama/README.md and the --help text below, fix the command, re-run.\n")
    print(help_txt[:6000])
    raise SystemExit("Stopped before export.")
print("All flags found in --help.")

# ===== 6) export (skipped if the .pte already exists) =====
os.makedirs(OUT, exist_ok=True)
if not os.path.exists(f"{OUT}/{PTE_NAME}"):
    cmd = (f"python -m examples.models.llama.export_llama "
           f"--model llama3_2 "
           f"--checkpoint {CK}/consolidated.00.pth --params {CK}/params.json "
           f"-kv --use_sdpa_with_kv_cache --xnnpack -d fp32 "
           f"--quantization_mode 8da4w --group_size {GROUP} --embedding-quantize 4,32 "
           f"--max_seq_length {CTX} "
           f"--metadata '{{\"get_bos_id\":128000,\"get_eos_ids\":[128009,128001]}}' "
           f"--output_name {PTE_NAME}")
    sh(f"{ENVP} cd {OUT} && {cmd}")
else:
    print("Export: .pte already exists, skipping.")

# ===== 7) tokenizer + save to Drive =====
sh(f"cp -v {CK}/tokenizer.model {OUT}/")
sh(f"ls -lh {OUT}")
save_outputs(pattern=f"{OUT}/*.pte {OUT}/tokenizer.model")
sh("ls -lh /content/drive/MyDrive/executorch_outputs/")
print("\nDone. Note the .pte and tokenizer sizes for your report.")

Using tag: v1.3.1
ExecuTorch source: already cloned.

$ uv venv --seed --python 3.12.14 /content/.llama_venv
Using CPython 3.12.14
Creating virtual environment with seed packages at: /content/.llama_venv
 + pip==26.3
Activate with: source /content/.llama_venv/bin/activate

$ export PATH=/content/.llama_venv/bin:$PATH VIRTUAL_ENV=/content/.llama_venv; cd /content/executorch_src; ./install_executorch.sh
2026-11-06 08:13:26,411 [ExecuTorch] INFO: All required submodules are present.
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/test/cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.3/192.3 MB 49.0 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 31.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 26.9 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 25.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/

RuntimeError: Command failed (1): /content/.llama_venv/bin/python -c 'import executorch; print("executorch", executorch.__version__)'

In [ ]:
# Where did ExecuTorch end up?
sh(f"{LPY} -m pip list 2>/dev/null | grep -i -E 'executorch|^torch' || echo 'NOT in llama venv'", check=False)
sh("python3 -m pip list 2>/dev/null | grep -i -E 'executorch|^torch' || echo 'NOT in system python'", check=False)
sh(f"{PY} -m pip list 2>/dev/null | grep -i -E 'executorch' || uv pip list --python {PY} | grep -i executorch", check=False)

# Which python do the install scripts actually see?
sh(f"{ENVP} which -a python; python -c 'import sys; print(sys.prefix)'", check=False)

# What install entry points does this release have?
sh(f"ls {LREPO} | grep -i install; ls {LREPO}/examples/models/llama | head -30", check=False)


$ /content/.llama_venv/bin/python -m pip list 2>/dev/null | grep -i -E 'executorch|^torch' || echo 'NOT in llama venv'
torch                  2.12.0+cpu
torchao                0.17.0+git02105d4
torchdata              0.11.0
torchtune              0.6.1

$ python3 -m pip list 2>/dev/null | grep -i -E 'executorch|^torch' || echo 'NOT in system python'
torch                                 2.11.0+cpu
torchao                               0.10.0
torchaudio                            2.11.0+cpu
torchcodec                            0.11.0+cpu
torchdata                             0.11.0
torchsummary                          1.5.1
torchtune                             0.6.1
torchvision                           0.26.0+cpu

$ /content/executorch_on_arm/.cpu_aot_venv/bin/python -m pip list 2>/dev/null | grep -i -E 'executorch' || uv pip list --python /content/executorch_on_arm/.cpu_aot_venv/bin/python | grep -i executorch
Using Python 3.12.14 environment at: .cpu_aot_venv
executorch          

0

In [ ]:
sh(f"{ENVP} python -m pip --version; python --version", check=False)

# Run the installer with a full log
sh(f"{ENVP} ./install_executorch.sh > /content/install.log 2>&1; echo exit=$?", check=False)

# What happened?
sh("tail -n 80 /content/install.log", check=False)
sh("grep -n -i -E 'error|failed|killed|no space|out of memory|Successfully installed' /content/install.log | tail -n 30", check=False)

# Is it in the venv now?
sh(f"{LPY} -c 'import executorch; print(executorch.__version__)'", check=False)
sh("df -h /content | tail -1; free -h | head -2", check=False)


$ export PATH=/content/.llama_venv/bin:$PATH VIRTUAL_ENV=/content/.llama_venv; cd /content/executorch_src; python -m pip --version; python --version
pip 26.3 from /content/.llama_venv/lib/python3.12/site-packages/pip (python 3.12)
Python 3.12.14

$ export PATH=/content/.llama_venv/bin:$PATH VIRTUAL_ENV=/content/.llama_venv; cd /content/executorch_src; ./install_executorch.sh > /content/install.log 2>&1; echo exit=$?
exit=0

$ tail -n 80 /content/install.log
    The ExecuTorch repo must be cloned into a directory named exactly
    `executorch`; found `executorch_src`.  See
    https://github.com/pytorch/executorch/issues/6475 for progress on a fix for
    this restriction.


  -- Configuring incomplete, errors occurred!
  2026-11-06 08:37:19,189 [INFO] restored HOME
  Traceback (most recent call last):
    File "/content/.llama_venv/lib/python3.12/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 406, in <module>
      main()
    File "/content/.llama_venv/lib

0

In [ ]:
import os, re, shutil, subprocess

# ===== helpers (self-contained) =====
def sh(cmd, check=True, quiet=False):
    if not quiet:
        print(f"\n$ {cmd}")
    p = subprocess.Popen(cmd, shell=True, executable="/bin/bash",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if not quiet:
            print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed ({p.returncode}): {cmd}")
    return p.returncode

def can_import(py, mod):
    return subprocess.run([py, "-c", f"import {mod}"], capture_output=True).returncode == 0

# ===== config =====
ET_VER   = "1.3.1"                        # <- version from your team's VERSIONS.md
CTX      = 1024                           # context length (512 or 1024 for the phone)
GROUP    = 32                             # quantization group size
LREPO    = "/content/executorch"          # MUST be named exactly "executorch"
OLDREPO  = "/content/executorch_src"
LVENV    = "/content/.llama_venv"
LPY      = f"{LVENV}/bin/python"
WDIR     = "/content/llama32_1b"
CK       = f"{WDIR}/original"
OUT      = "/content/export"
PTE_NAME = f"llama3_2_1b_8da4w_{CTX}.pte"
ENVP     = (f"export PATH={LVENV}/bin:$PATH VIRTUAL_ENV={LVENV} "
            f"CMAKE_BUILD_PARALLEL_LEVEL=4; cd {LREPO};")

# ===== HF token + Drive =====
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("HF_TOKEN not loaded:", e)
assert os.environ.get("HF_TOKEN"), "Add HF_TOKEN in Colab Secrets and enable Notebook access."

try:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted:", e)

def save_outputs(pattern="*.pte tokenizer*", dest="/content/drive/MyDrive/executorch_outputs"):
    os.makedirs(dest, exist_ok=True)
    sh(f"cp -v {pattern} {dest}/", check=False)

# ===== 1) find the git tag for your release =====
tags = subprocess.check_output(
    "git ls-remote --tags https://github.com/pytorch/executorch.git | awk '{print $2}'",
    shell=True, text=True).splitlines()
cands = [t.replace("refs/tags/", "") for t in tags
         if re.fullmatch(rf"refs/tags/v?{re.escape(ET_VER)}", t)]
assert cands, f"No tag found for {ET_VER}. Check the release name and set ET_VER."
TAG = cands[0]
print("Using tag:", TAG)

# ===== 2) clone into a folder named exactly "executorch" =====
if os.path.isdir(OLDREPO) and not os.path.isdir(LREPO):
    shutil.move(OLDREPO, LREPO)
    print("Renamed executorch_src -> executorch")
if not os.path.isdir(LREPO):
    sh(f"git clone --branch {TAG} --depth 1 --recurse-submodules --shallow-submodules "
       f"https://github.com/pytorch/executorch.git {LREPO}")
else:
    print("ExecuTorch source: already present.")

# ===== 3) separate venv + install from source =====
if not os.path.exists(LPY):
    sh("pip install -q uv", quiet=True)
    sh(f"uv venv --seed --python 3.12.14 {LVENV}")

if not can_import(LPY, "executorch"):
    sh(f"rm -rf {LREPO}/pip-out", check=False)            # clear any failed build
    print("Installing ExecuTorch from source (20-40 min, keep the tab open)...")
    sh(f"{ENVP} ./install_executorch.sh > /content/install.log 2>&1", check=False, quiet=True)
    if not can_import(LPY, "executorch"):
        sh("grep -n -i -E 'CMake Error|Failed building|Killed|No space' /content/install.log | tail -n 20", check=False)
        sh("tail -n 40 /content/install.log", check=False)
        raise SystemExit("ExecuTorch install failed. Paste the lines above.")
    sh(f"{ENVP} bash examples/models/llama/install_requirements.sh > /content/install_llama.log 2>&1",
       check=False, quiet=True)
    if not can_import(LPY, "executorch"):
        sh("tail -n 40 /content/install_llama.log", check=False)
        raise SystemExit("Llama requirements broke the install. Paste the lines above.")
else:
    print("Llama venv: ExecuTorch already installed.")
sh(f"{LPY} -c 'import executorch; print(\"executorch\", executorch.__version__)'")

# ===== 4) download original-format Llama 3.2 1B Instruct weights =====
if not os.path.exists(f"{CK}/consolidated.00.pth"):
    sh(f"{LPY} -m pip install -q huggingface_hub", quiet=True, check=False)
    sh(f"{LPY} -c \"from huggingface_hub import snapshot_download; "
       f"snapshot_download('meta-llama/Llama-3.2-1B-Instruct', "
       f"allow_patterns=['original/*'], local_dir='{WDIR}')\"")
else:
    print("Weights: already downloaded.")
sh(f"ls -lh {CK}")

# ===== 5) verify flags exist in THIS release before exporting =====
help_txt = subprocess.run(f"{ENVP} python -m examples.models.llama.export_llama --help",
                          shell=True, capture_output=True, text=True).stdout
need = ["--model", "--checkpoint", "--params", "-kv", "--use_sdpa_with_kv_cache", "--xnnpack",
        "--quantization_mode", "--group_size", "--embedding-quantize", "--max_seq_length",
        "--metadata", "--output_name"]
missing = [f for f in need if f not in help_txt]
if missing:
    print("Flags NOT found in --help for this release:", missing)
    print(help_txt[:6000])
    raise SystemExit("Stopped before export. Paste this output.")
print("All flags found in --help.")

# ===== 6) export =====
os.makedirs(OUT, exist_ok=True)
if not os.path.exists(f"{OUT}/{PTE_NAME}"):
    cmd = (f"python -m examples.models.llama.export_llama "
           f"--model llama3_2 "
           f"--checkpoint {CK}/consolidated.00.pth --params {CK}/params.json "
           f"-kv --use_sdpa_with_kv_cache --xnnpack -d fp32 "
           f"--quantization_mode 8da4w --group_size {GROUP} --embedding-quantize 4,32 "
           f"--max_seq_length {CTX} "
           f"--metadata '{{\"get_bos_id\":128000,\"get_eos_ids\":[128009,128001]}}' "
           f"--output_name {PTE_NAME}")
    sh(f"{ENVP} cd {OUT} && {cmd}")
else:
    print("Export: .pte already exists, skipping.")

# ===== 7) tokenizer + save to Drive =====
sh(f"cp -v {CK}/tokenizer.model {OUT}/")
sh(f"ls -lh {OUT}")
save_outputs(pattern=f"{OUT}/*.pte {OUT}/tokenizer.model")
sh("ls -lh /content/drive/MyDrive/executorch_outputs/")
print("\nDone. Note the .pte and tokenizer sizes for your report.")

Using tag: v1.3.1
Renamed executorch_src -> executorch
ExecuTorch source: already present.

$ rm -rf /content/executorch/pip-out
Installing ExecuTorch from source (20-40 min, keep the tab open)...

$ grep -n -i -E 'CMake Error|Failed building|Killed|No space' /content/install.log | tail -n 20
117:  ERROR: Failed building wheel for pytorch_tokenizers

$ tail -n 40 /content/install.log
  Created wheel for torchao: filename=torchao-0.17.0+git02105d4-py3-none-any.whl size=1185795 sha256=5e461a62c6b19dc276fb800b41bc1a70299d95716031eaa0106a6598f6417849
  Stored in directory: /tmp/pip-ephem-wheel-cache-7frsfsco/wheels/0e/36/70/fc318484002137bbdd30c7de066525cdb6b2054d8602ede207
  error: subprocess-exited-with-error
  
  × Building wheel for pytorch_tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pytorch_tokenizers
Success

SystemExit: ExecuTorch install failed. Paste the lines above.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# 1) Pin setuptools to what torch requires
sh(f"{LPY} -m pip install -q 'setuptools<82'", check=False)

# 2) Delete every stale build folder / CMake cache that remembers executorch_src
sh(f"rm -rf {LREPO}/pip-out {LREPO}/extension/llm/tokenizers/build {LREPO}/third-party/ao/build", check=False)
sh(f"find {LREPO} -name CMakeCache.txt -not -path '*/third-party/*/third_party/*' -print -delete", check=False)
sh(f"find {LREPO} -type d -name CMakeFiles -prune -exec rm -rf {{}} +", check=False)
sh(f"grep -rl 'executorch_src' {LREPO} --include=CMakeCache.txt 2>/dev/null | head", check=False)  # should print nothing

# 3) Re-run the installer (slow: 20-40 min, keep the tab open)
sh(f"{ENVP} ./install_executorch.sh > /content/install.log 2>&1", check=False, quiet=True)

# 4) Check the result
sh("grep -n -i -E 'CMake Error|Failed building|Killed|No space' /content/install.log | tail -n 20", check=False)
sh(f"{LPY} -c 'import executorch; print(\"executorch\", executorch.__version__)'", check=False)


$ /content/.llama_venv/bin/python -m pip install -q 'setuptools<82'


KeyboardInterrupt: 